In [1]:
"""Example using the ideas from IBM's 'Construct circuits' page.

Part 1: build a Bell state from sub-circuits (compose) and sample it.
Part 2: a parameterized circuit, built once and swept over many angles.
"""
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2

backend = AerSimulator()
sampler = SamplerV2()

# ---------- Part 1: Bell state from two sub-circuits ----------
superpose = QuantumCircuit(2, name="superpose")
superpose.h(0)

entangle = QuantumCircuit(2, name="entangle")
entangle.cx(0, 1)

bell = superpose.compose(entangle)   # compose returns a new circuit
bell.measure_all()                   # Sampler requires measurements
print(bell.draw("text"))
print("depth:", bell.depth())

counts = sampler.run([bell], shots=1000).result()[0].data.meas.get_counts()
print("Bell counts:", counts)        # expect only '00' and '11'

# ---------- Part 2: parameterized sweep ----------
theta = Parameter("theta")
qc = QuantumCircuit(1)
qc.rx(theta, 0)
qc.measure_all()

# transpile once, outside the loop
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
isa = pm.run(qc)

angles = np.linspace(0, np.pi, 7)
job = sampler.run([(isa, [[a] for a in angles])], shots=2000)
data = job.result()[0].data.meas

print("\nangle   P(1)")
for a, i in zip(angles, range(len(angles))):
    c = data[i].get_counts()
    print(f"{a:5.2f}   {c.get('1', 0) / 2000:.3f}")   # theory: sin^2(a/2)

        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 
depth: 3
Bell counts: {'00': 518, '11': 482}

angle   P(1)
 0.00   0.000
 0.52   0.065
 1.05   0.268
 1.57   0.494
 2.09   0.741
 2.62   0.935
 3.14   1.000
